# 14 FS2 Staged Ablation Evaluation

This notebook is the combined evaluation layer for the upgraded `FS2` staged ablation workflow across `LEAR` and `XGBoost`.

It has two jobs:
- synthesize the baseline `FS2` staged-ablation results across both models
- optionally show a revised candidate lineage after notebook `15` reruns a pruned or redesigned parent


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


In [ ]:
from IPython.display import Markdown

from hourly_da.core.ablation_blocks import (
    SCHEME_NAME_LAYER_1,
    SCHEME_NAME_LAYER_2,
    SCHEME_NAME_STAGE_A,
    supported_fs2_layer2_target_blocks,
)
from hourly_da.notebook_support import (
    ablation_effect_label,
    annotate_ablation_metric_slice,
    apply_notebook_display_defaults,
    load_fs2_ablation_report_dataset,
    select_feature_family_metric_slice,
    summarize_ablation_cross_model,
)

apply_notebook_display_defaults()

RUN_DIAGNOSTICS = False
PRIMARY_SPLIT = "validation"
PRIMARY_METRIC = "mae"
VISIBLE_REPORTING_LEVELS = ("d_only", "stitched_all_horizon")

_combined_layer2_targets = supported_fs2_layer2_target_blocks()
FS2_BASELINE_PARENT_SPECS = [{'parent_run_label': 'lear_fs2_benchmark', 'model_family': 'lear', 'model_label': 'LEAR', 'model_name': 'lear_fs2'}, {'parent_run_label': 'xgboost_fs2_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost', 'model_name': 'xgboost_fs2'}]
FS2_BASELINE_SCHEME_REQUESTS = [
    {"scheme_name": SCHEME_NAME_STAGE_A, "target_block": None},
    {"scheme_name": SCHEME_NAME_LAYER_1, "target_block": None},
]
FS2_BASELINE_SCHEME_REQUESTS.extend(
    {"scheme_name": SCHEME_NAME_LAYER_2, "target_block": target_block}
    for target_block in _combined_layer2_targets
)

FS2_BASELINE_REPORT = load_fs2_ablation_report_dataset(
    output_root,
    config,
    parent_specs=FS2_BASELINE_PARENT_SPECS,
    scheme_requests=FS2_BASELINE_SCHEME_REQUESTS,
)
FS2_BASELINE_AVAILABILITY = FS2_BASELINE_REPORT["availability"]
FS2_BASELINE_BUNDLES = FS2_BASELINE_REPORT["bundles"]
FS2_BASELINE_SUMMARY = FS2_BASELINE_REPORT["summary"]
FS2_BASELINE_BY_ORIGIN = FS2_BASELINE_REPORT["by_origin"]


In [ ]:
from IPython.display import Markdown

from hourly_da.core.ablation_blocks import (
    SCHEME_NAME_LAYER_1,
    SCHEME_NAME_LAYER_2,
    SCHEME_NAME_STAGE_A,
    supported_fs2_layer2_target_blocks,
)
from hourly_da.notebook_support import (
    ablation_effect_label,
    annotate_ablation_metric_slice,
    apply_notebook_display_defaults,
    load_fs2_ablation_report_dataset,
    select_feature_family_metric_slice,
    summarize_ablation_cross_model,
)

apply_notebook_display_defaults()

RUN_DIAGNOSTICS = False
PRIMARY_SPLIT = "validation"
PRIMARY_METRIC = "mae"
VISIBLE_REPORTING_LEVELS = ("d_only", "stitched_all_horizon")

_combined_layer2_targets = supported_fs2_layer2_target_blocks()
FS2_CANDIDATE_PARENT_SPECS = [{'parent_run_label': 'lear_fs2_pruned_candidate_benchmark', 'model_family': 'lear', 'model_label': 'LEAR candidate', 'model_name': 'lear_fs2'}, {'parent_run_label': 'xgboost_fs2_pruned_candidate_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost candidate', 'model_name': 'xgboost_fs2'}]
FS2_CANDIDATE_SCHEME_REQUESTS = [
    {"scheme_name": SCHEME_NAME_STAGE_A, "target_block": None},
    {"scheme_name": SCHEME_NAME_LAYER_1, "target_block": None},
]
FS2_CANDIDATE_SCHEME_REQUESTS.extend(
    {"scheme_name": SCHEME_NAME_LAYER_2, "target_block": target_block}
    for target_block in _combined_layer2_targets
)

FS2_CANDIDATE_REPORT = load_fs2_ablation_report_dataset(
    output_root,
    config,
    parent_specs=FS2_CANDIDATE_PARENT_SPECS,
    scheme_requests=FS2_CANDIDATE_SCHEME_REQUESTS,
)
FS2_CANDIDATE_AVAILABILITY = FS2_CANDIDATE_REPORT["availability"]
FS2_CANDIDATE_BUNDLES = FS2_CANDIDATE_REPORT["bundles"]
FS2_CANDIDATE_SUMMARY = FS2_CANDIDATE_REPORT["summary"]
FS2_CANDIDATE_BY_ORIGIN = FS2_CANDIDATE_REPORT["by_origin"]


## Baseline Compatibility Gate

This is the active thesis-grade baseline synthesis. Only compatible staged-ablation bundles are included.


In [ ]:
availability_view = FS2_BASELINE_AVAILABILITY[
    [
        "model_label",
        "scheme_name",
        "layer_name",
        "target_block",
        "availability_status",
        "latest_timestamp_label",
        "status_note",
    ]
].rename(
    columns={
        "model_label": "Model",
        "scheme_name": "Scheme",
        "layer_name": "Layer",
        "target_block": "Layer 2 target",
        "availability_status": "Status",
        "latest_timestamp_label": "Latest timestamp",
        "status_note": "Status note",
    }
)
display(availability_view.style.hide(axis="index"))

display(
    Markdown(
        "Active synthesis scope for baseline FS2 lineage: only compatible staged-ablation bundles are included below. "
        "Saved aggregates with stale scheme hashes, stale taxonomy hashes, or invalid stored preflight are excluded."
    )
)


## Baseline Stage A Across Both Models


In [ ]:
scheme_summary = FS2_BASELINE_SUMMARY[
    FS2_BASELINE_SUMMARY["scheme_name"].astype(str) == 'stage_a_top_level'
].copy()

if scheme_summary.empty:
    print("No compatible saved results are available for this layer yet.")
else:
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            scheme_summary,
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        model_view = annotated[
            [
                "model_family",
                "feature_family",
                "parent_value",
                "child_value",
                "delta",
                "relative_delta",
                "effect_label",
            ]
        ].rename(
            columns={
                "model_family": "Model family",
                "feature_family": "Block",
                "parent_value": "Parent value",
                "child_value": "Child value",
                "delta": "Delta",
                "relative_delta": "Relative delta",
                "effect_label": "Interpretation",
            }
        )
        display(
            model_view.style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        combined = summarize_ablation_cross_model(metric_slice)
        display(
            combined.rename(
                columns={
                    "feature_family": "Block",
                    "models_available": "Models available",
                    "model_count": "Model count",
                    "mean_delta": "Mean delta",
                    "mean_relative_delta": "Mean relative delta",
                    "combined_label": "Combined classification",
                }
            ).style
            .format(
                {
                    "Mean delta": "{:+.4f}",
                    "Mean relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )


## Baseline Layer 1 Across Both Models


In [ ]:
scheme_summary = FS2_BASELINE_SUMMARY[
    FS2_BASELINE_SUMMARY["scheme_name"].astype(str) == 'layer1_mutually_exclusive'
].copy()

if scheme_summary.empty:
    print("No compatible saved results are available for this layer yet.")
else:
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            scheme_summary,
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        model_view = annotated[
            [
                "model_family",
                "feature_family",
                "parent_value",
                "child_value",
                "delta",
                "relative_delta",
                "effect_label",
            ]
        ].rename(
            columns={
                "model_family": "Model family",
                "feature_family": "Block",
                "parent_value": "Parent value",
                "child_value": "Child value",
                "delta": "Delta",
                "relative_delta": "Relative delta",
                "effect_label": "Interpretation",
            }
        )
        display(
            model_view.style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        combined = summarize_ablation_cross_model(metric_slice)
        display(
            combined.rename(
                columns={
                    "feature_family": "Block",
                    "models_available": "Models available",
                    "model_count": "Model count",
                    "mean_delta": "Mean delta",
                    "mean_relative_delta": "Mean relative delta",
                    "combined_label": "Combined classification",
                }
            ).style
            .format(
                {
                    "Mean delta": "{:+.4f}",
                    "Mean relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )


## Baseline Layer 2 Follow-Up


In [ ]:
available_layer2 = FS2_BASELINE_SUMMARY[
    FS2_BASELINE_SUMMARY["scheme_name"].astype(str) == "layer2_subgroups"
].copy()

if available_layer2.empty:
    print("No compatible Layer 2 subgroup results are available yet.")
else:
    for target_block, group in available_layer2.groupby("target_block", dropna=False):
        display(Markdown(f"### {str(target_block).replace('_', ' ').title()}"))
        for reporting_level in VISIBLE_REPORTING_LEVELS:
            metric_slice = select_feature_family_metric_slice(
                group,
                split=PRIMARY_SPLIT,
                reporting_level=reporting_level,
                metric=PRIMARY_METRIC,
            )
            if metric_slice.empty:
                continue
            annotated = annotate_ablation_metric_slice(metric_slice)
            display(Markdown(f"#### {reporting_level.replace('_', ' ').title()}"))
            display(
                annotated[
                    [
                        "model_family",
                        "feature_family",
                        "delta",
                        "relative_delta",
                        "effect_label",
                    ]
                ]
                .rename(
                    columns={
                        "model_family": "Model family",
                        "feature_family": "Subgroup",
                        "delta": "Delta",
                        "relative_delta": "Relative delta",
                        "effect_label": "Interpretation",
                    }
                )
                .style
                .format({"Delta": "{:+.4f}", "Relative delta": "{:+.3%}"})
                .hide(axis="index")
            )


## Baseline Supportive Diagnostics


In [ ]:
from hourly_da.notebook_support import compute_fs2_parent_block_diagnostics

if not RUN_DIAGNOSTICS:
    print("Diagnostics are disabled for this notebook.")
else:
    for parent_spec in FS2_BASELINE_PARENT_SPECS:
        display(Markdown(f"### {parent_spec['model_label']}"))
        diagnostics = compute_fs2_parent_block_diagnostics(
            config,
            parent_run_label=str(parent_spec["parent_run_label"]),
            model_family=str(parent_spec["model_family"]),
            split_name="validation",
        )
        display(diagnostics["reference"])
        for diagnostic_block in diagnostics["tables"]:
            display(
                Markdown(
                    f"#### {diagnostic_block['branch_name'].replace('_', ' ').title()} / "
                    f"{diagnostic_block['diagnostic_type'].replace('_', ' ').title()}"
                )
            )
            table = diagnostic_block["table"]
            if diagnostic_block["diagnostic_type"] == "lear_coefficients":
                display(
                    table.style
                    .format({"sum_abs_standardized_coefficient": "{:.4f}"})
                    .hide(axis="index")
                )
            else:
                display(
                    table.style
                    .format({"total_gain": "{:.4f}", "total_split_count": "{:.0f}"})
                    .hide(axis="index")
                )


## Candidate Lineage Delta Check

After notebook `15` runs a revised parent, rerun this notebook to see whether the candidate benchmark improved and whether the staged-ablation story changed.


In [ ]:
baseline_specs = [{'parent_run_label': 'lear_fs2_benchmark', 'model_family': 'lear', 'model_label': 'LEAR', 'model_name': 'lear_fs2'}, {'parent_run_label': 'xgboost_fs2_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost', 'model_name': 'xgboost_fs2'}]
candidate_specs = [{'parent_run_label': 'lear_fs2_pruned_candidate_benchmark', 'model_family': 'lear', 'model_label': 'LEAR candidate', 'model_name': 'lear_fs2'}, {'parent_run_label': 'xgboost_fs2_pruned_candidate_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost candidate', 'model_name': 'xgboost_fs2'}]
comparison_rows = []

for baseline_spec, candidate_spec in zip(baseline_specs, candidate_specs):
    baseline_run_dir = latest_run_or_none(str(baseline_spec["parent_run_label"]))
    candidate_run_dir = latest_run_or_none(str(candidate_spec["parent_run_label"]))
    if baseline_run_dir is None:
        continue
    baseline_metrics = load_csv(baseline_run_dir, "metrics_by_reporting_level.csv")
    baseline_metrics = baseline_metrics[baseline_metrics["model"].astype(str) == str(baseline_spec["model_name"])].copy()
    if baseline_metrics.empty:
        continue
    candidate_metrics = baseline_metrics.iloc[0:0].copy()
    if candidate_run_dir is not None:
        candidate_metrics = load_csv(candidate_run_dir, "metrics_by_reporting_level.csv")
        candidate_metrics = candidate_metrics[candidate_metrics["model"].astype(str) == str(candidate_spec["model_name"])].copy()

    for split_name in ("validation", "test"):
        for reporting_level in ("d_only", "stitched_all_horizon"):
            base_row = baseline_metrics[
                (baseline_metrics["dataset_split"].astype(str) == split_name)
                & (baseline_metrics["reporting_level"].astype(str) == reporting_level)
            ].head(1)
            candidate_row = candidate_metrics[
                (candidate_metrics["dataset_split"].astype(str) == split_name)
                & (candidate_metrics["reporting_level"].astype(str) == reporting_level)
            ].head(1)
            comparison_rows.append(
                {
                    "Model family": baseline_spec["model_family"],
                    "Model label": baseline_spec["model_label"],
                    "Dataset split": split_name,
                    "Reporting level": reporting_level,
                    "Baseline run": baseline_spec["parent_run_label"],
                    "Candidate run": candidate_spec["parent_run_label"] if candidate_run_dir is not None else "",
                    "Baseline MAE": float(base_row["mae"].iloc[0]) if not base_row.empty else float("nan"),
                    "Candidate MAE": float(candidate_row["mae"].iloc[0]) if not candidate_row.empty else float("nan"),
                    "MAE delta (candidate - baseline)": (
                        float(candidate_row["mae"].iloc[0]) - float(base_row["mae"].iloc[0])
                        if (not base_row.empty and not candidate_row.empty)
                        else float("nan")
                    ),
                    "Baseline rMAE": float(base_row["rmae_vs_official_naive"].iloc[0]) if not base_row.empty else float("nan"),
                    "Candidate rMAE": float(candidate_row["rmae_vs_official_naive"].iloc[0]) if not candidate_row.empty else float("nan"),
                }
            )

comparison_frame = pd.DataFrame(comparison_rows)
if comparison_frame.empty:
    print("No baseline benchmark rows were available for the candidate comparison.")
else:
    display(
        comparison_frame.style
        .format(
            {
                "Baseline MAE": "{:.4f}",
                "Candidate MAE": "{:.4f}",
                "MAE delta (candidate - baseline)": "{:+.4f}",
                "Baseline rMAE": "{:.4f}",
                "Candidate rMAE": "{:.4f}",
            }
        )
        .hide(axis="index")
    )


## Candidate Compatibility Gate


In [ ]:
availability_view = FS2_CANDIDATE_AVAILABILITY[
    [
        "model_label",
        "scheme_name",
        "layer_name",
        "target_block",
        "availability_status",
        "latest_timestamp_label",
        "status_note",
    ]
].rename(
    columns={
        "model_label": "Model",
        "scheme_name": "Scheme",
        "layer_name": "Layer",
        "target_block": "Layer 2 target",
        "availability_status": "Status",
        "latest_timestamp_label": "Latest timestamp",
        "status_note": "Status note",
    }
)
display(availability_view.style.hide(axis="index"))

display(
    Markdown(
        "Active synthesis scope for candidate FS2 lineage: only compatible staged-ablation bundles are included below. "
        "Saved aggregates with stale scheme hashes, stale taxonomy hashes, or invalid stored preflight are excluded."
    )
)


## Candidate Stage A Across Both Models


In [ ]:
scheme_summary = FS2_CANDIDATE_SUMMARY[
    FS2_CANDIDATE_SUMMARY["scheme_name"].astype(str) == 'stage_a_top_level'
].copy()

if scheme_summary.empty:
    print("No compatible saved results are available for this layer yet.")
else:
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            scheme_summary,
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        model_view = annotated[
            [
                "model_family",
                "feature_family",
                "parent_value",
                "child_value",
                "delta",
                "relative_delta",
                "effect_label",
            ]
        ].rename(
            columns={
                "model_family": "Model family",
                "feature_family": "Block",
                "parent_value": "Parent value",
                "child_value": "Child value",
                "delta": "Delta",
                "relative_delta": "Relative delta",
                "effect_label": "Interpretation",
            }
        )
        display(
            model_view.style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        combined = summarize_ablation_cross_model(metric_slice)
        display(
            combined.rename(
                columns={
                    "feature_family": "Block",
                    "models_available": "Models available",
                    "model_count": "Model count",
                    "mean_delta": "Mean delta",
                    "mean_relative_delta": "Mean relative delta",
                    "combined_label": "Combined classification",
                }
            ).style
            .format(
                {
                    "Mean delta": "{:+.4f}",
                    "Mean relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )


## Candidate Layer 1 Across Both Models


In [ ]:
scheme_summary = FS2_CANDIDATE_SUMMARY[
    FS2_CANDIDATE_SUMMARY["scheme_name"].astype(str) == 'layer1_mutually_exclusive'
].copy()

if scheme_summary.empty:
    print("No compatible saved results are available for this layer yet.")
else:
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            scheme_summary,
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        model_view = annotated[
            [
                "model_family",
                "feature_family",
                "parent_value",
                "child_value",
                "delta",
                "relative_delta",
                "effect_label",
            ]
        ].rename(
            columns={
                "model_family": "Model family",
                "feature_family": "Block",
                "parent_value": "Parent value",
                "child_value": "Child value",
                "delta": "Delta",
                "relative_delta": "Relative delta",
                "effect_label": "Interpretation",
            }
        )
        display(
            model_view.style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        combined = summarize_ablation_cross_model(metric_slice)
        display(
            combined.rename(
                columns={
                    "feature_family": "Block",
                    "models_available": "Models available",
                    "model_count": "Model count",
                    "mean_delta": "Mean delta",
                    "mean_relative_delta": "Mean relative delta",
                    "combined_label": "Combined classification",
                }
            ).style
            .format(
                {
                    "Mean delta": "{:+.4f}",
                    "Mean relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )
